CSV AND EXCEL FILES INSGESTION AND PARSING

In [7]:
import pandas as pd
import os

In [3]:
os.makedirs("data/structured_files",exist_ok=True)

creating  temp data and saving as csv

In [4]:
data = {
    'Product':['Laptop','Mouse','Keyboard','Monitor','Webcam'],
    'Category': ['Electronics','Accessories','Accessories','Electronics','Electronics'],
    'Price':[999.99,29.99,79.99,299.99,89.99],
    'Stock':[50,200,150,75,100],
    'Description':[
        'High-performance laptop with 16GB RAM and 512GB SSD',
        'Wirless optical mouse with ergonomic design',
        'Mechanical keyboard with RCB backlighting',
        '27-inch 4k monitor with HDR support',
        '1080p webcam with noise cancellation'
    ]
}

# saving as csv
df = pd.DataFrame(data)
df.to_csv('data/structured_files/products.csv',index=False)

saving same data as an excel file

In [6]:
with pd.ExcelWriter('data/structured_files/inventory.xlsx') as writer:
    df.to_excel(writer,sheet_name="Prodcuts",index=False)

    # one more sheet
    summary_data ={
        'Category':['Electronics','Accessories'],
        'Total_Items':[3,2],
        'Total_Value':[1389.91,109.98]
    }
    pd.DataFrame(summary_data).to_excel(writer,sheet_name="Summary",index=False)

libraries for loading csv files

In [1]:
from langchain_community.document_loaders import CSVLoader
from langchain_community.document_loaders import UnstructuredCSVLoader

CSVLoader

In [ ]:
csv_loader =  CSVLoader(
    file_path = 'data/structured_files/products.csv',
    encoding = 'utf-8',
    csv_args = {
        'delimiter': ',',
        'quotechar': '"',
    } 
)
csv_docs = csv_loader.load()
print(f"No of docs loaded: {len(csv_docs)}")
print("First document")
print(f"Content: {csv_docs[0].page_content}")
print(f"Metadata: {csv_docs[0].metadata}")

No of docs loaded: 5
First document
Content: Product: Laptop
Category: Electronics
Price: 999.99
Stock: 50
Description: High-performance laptop with 16GB RAM and 512GB SSD
Metadata: {'source': 'data/structured_files/products.csv', 'row': 0}


Custom CSV Processor 

In [5]:
from typing import List
from langchain_core.documents import Document

def process_csv(filepath: str) -> List[Document]:
    df = pd.read_csv(filepath)
    documents=[]

    for idx,row in df.iterrows():
        content = f"""Product Information:
        Name: {row['Product']}
        Categoey: {row['Category']}
        Price: ${row['Price']}
        Stock: {row['Stock']} units
        Description: {row['Description']}
        """

        doc = Document(
            page_content=content,
            metadata = {
                'source': filepath,
                'row_index': idx,
                'product_name': row['Product'],
                'data_type': 'product_info xyz',
                'category': row['Category']
            }
        )
        documents.append(doc)
    return documents


In [8]:
process_csv('data/structured_files/products.csv')

[Document(metadata={'source': 'data/structured_files/products.csv', 'row_index': 0, 'product_name': 'Laptop', 'data_type': 'product_info xyz', 'category': 'Electronics'}, page_content='Product Information:\n        Name: Laptop\n        Categoey: Electronics\n        Price: $999.99\n        Stock: 50 units\n        Description: High-performance laptop with 16GB RAM and 512GB SSD\n        '),
 Document(metadata={'source': 'data/structured_files/products.csv', 'row_index': 1, 'product_name': 'Mouse', 'data_type': 'product_info xyz', 'category': 'Accessories'}, page_content='Product Information:\n        Name: Mouse\n        Categoey: Accessories\n        Price: $29.99\n        Stock: 200 units\n        Description: Wirless optical mouse with ergonomic design\n        '),
 Document(metadata={'source': 'data/structured_files/products.csv', 'row_index': 2, 'product_name': 'Keyboard', 'data_type': 'product_info xyz', 'category': 'Accessories'}, page_content='Product Information:\n        Nam

Excel processing

In [9]:
# METHOD 1 USING PANDAS
def process_excel_wpandas(filepath: str) -> List[Document]:
    documents=[]
    file = pd.ExcelFile(filepath)

    for sheet in file.sheet_names:
        df = pd.read_excel(filepath, sheet_name=sheet)

        # building doc for each sheet
        sheet_content = f"Sheet: {sheet}\n"
        sheet_content += f"Columns: {', '.join(df.columns)}\n"
        sheet_content += f"Rows: {len(df)}\n\n"
        sheet_content += df.to_string(index=False)

        doc = Document(
            page_content=sheet_content,
            metadata = {
                'source': filepath,
                'sheet_name': sheet,
                'num_rows': len(df),
                'num_columns': len(df.columns),
                'data_type': 'excel_sheet'
            }
        )
        documents.append(doc)
    return documents

In [10]:
excel_docs = process_excel_wpandas('data/structured_files/inventory.xlsx')
print(f"No of sheets in excel file: {len(excel_docs)}")
excel_docs

No of sheets in excel file: 2


[Document(metadata={'source': 'data/structured_files/inventory.xlsx', 'sheet_name': 'Prodcuts', 'num_rows': 5, 'num_columns': 5, 'data_type': 'excel_sheet'}, page_content='Sheet: Prodcuts\nColumns: Product, Category, Price, Stock, Description\nRows: 5\n\n Product    Category  Price  Stock                                         Description\n  Laptop Electronics 999.99     50 High-performance laptop with 16GB RAM and 512GB SSD\n   Mouse Accessories  29.99    200         Wirless optical mouse with ergonomic design\nKeyboard Accessories  79.99    150           Mechanical keyboard with RCB backlighting\n Monitor Electronics 299.99     75                 27-inch 4k monitor with HDR support\n  Webcam Electronics  89.99    100                1080p webcam with noise cancellation'),
 Document(metadata={'source': 'data/structured_files/inventory.xlsx', 'sheet_name': 'Summary', 'num_rows': 2, 'num_columns': 3, 'data_type': 'excel_sheet'}, page_content='Sheet: Summary\nColumns: Category, Total_Ite

In [ ]:
# METHOD 2 UNSTRUCTURED EXCEL LOADER

In [14]:
from langchain_community.document_loaders import UnstructuredExcelLoader

In [15]:
try:
    excel_loader = UnstructuredExcelLoader(
        'data/structured_files/inventory.xlsx',
        mode = "elements"
    )
    unstructured_docs = excel_loader.load()
    unstructured_docs
except Exception as e:
    print(f"Error loading file: {e}")


Error loading file: No module named 'msoffcrypto'
